In [1]:
import numpy as np

states = ['E', '5', 'I']
state_transitions = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

def compute_log(x):
    return -np.inf if x == 0 else np.log(x)

def calculate_path_log_prob(state_path, dna_sequence):
    if len(state_path) != len(dna_sequence):
        raise ValueError("State path and DNA sequence length mismatch")
    
    total_log_prob = 0.0
    previous_state = 'Start'
    
    for position in range(len(dna_sequence)):
        current_state = state_path[position]
        nucleotide = dna_sequence[position]
        
        transition_prob = state_transitions[previous_state].get(current_state, 0)
        emission_prob = emission_probs[current_state].get(nucleotide, 0)
        
        total_log_prob += compute_log(transition_prob) + compute_log(emission_prob)
        previous_state = current_state
    
    if previous_state == 'I':
        total_log_prob += compute_log(state_transitions['I']['End'])
    
    return np.round(total_log_prob, 2)

def find_optimal_path(dna_sequence):
    sequence_length = len(dna_sequence)
    probability_table = np.full((sequence_length, len(states)), -np.inf)
    path_history = {s: [] for s in states}
    
    probability_table[0, 0] = (
        compute_log(state_transitions['Start']['E']) + 
        compute_log(emission_probs['E'][dna_sequence[0]]))
    path_history['E'] = ['E']
    
    state_idx = {s: i for i, s in enumerate(states)}
    
    for position in range(1, sequence_length):
        current_paths = {s: [] for s in states}
        
        for current_state in states:
            max_prob = -np.inf
            best_previous_state = None
            
            for previous_state in states:
                prev_idx = state_idx[previous_state]
                if probability_table[position-1, prev_idx] == -np.inf:
                    continue
                    
                if current_state in state_transitions.get(previous_state, {}):
                    transition_log = compute_log(state_transitions[previous_state][current_state])
                    emission_log = compute_log(emission_probs[current_state][dna_sequence[position]])
                    total_prob = probability_table[position-1, prev_idx] + transition_log + emission_log
                    
                    if total_prob > max_prob:
                        max_prob = total_prob
                        best_previous_state = previous_state
            
            if best_previous_state is not None:
                current_idx = state_idx[current_state]
                probability_table[position, current_idx] = max_prob
                current_paths[current_state] = path_history[best_previous_state] + [current_state]
        
        path_history = current_paths
    
    final_idx = np.argmax(probability_table[-1])
    final_state = states[final_idx]
    return ''.join(path_history[final_state]), np.round(probability_table[-1, final_idx], 2)

example_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
example_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

print(f"Path probability: {calculate_path_log_prob(example_path, example_sequence)}")
optimal_path, optimal_prob = find_optimal_path(example_sequence)
print(f"Optimal path: {optimal_path}")
print(f"Optimal Path probability: {optimal_prob}")

Path probability: -41.22
Optimal path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Optimal Path probability: -38.68
